In [1]:
from pathlib import Path
import ims
from skimage.io import imshow
from sympy.strategies.branch import condition

import preprocess

# Step 0: Define all necessary directories.
input_dir = Path("./data/IMS_Olive_Oil_Geography/Test/")
output_dir = Path("./output/Jupyter_Test/")

# Check and create directories if necessary.
if not output_dir.exists():
    output_dir.mkdir(parents=True, exist_ok=True)

# Step 1: Preprocess the .mea files
# - RIP Interpolation (for each dataset)
# - Background noise reduction
# - Gaussian smoothing

# Load the test dataset and interpolate to a common RIP
dataset = ims.Dataset.read_mea(input_dir)
dataset.interp_riprel()

# Get the first test spectrum.
spectrum = dataset[0]

In [7]:
# =========================================== JUST PLOTTING THE CUTTED RAW SPECTRUM ===========================================
import plot
spectrum.cut_dt(start=1.02, stop=1.2)
spectrum.cut_rt(start=100, stop=140)

# Plot the raw spectrum.
plot.single_gcims_plot(spectrum=spectrum,
                       plot_dir=output_dir,
                       condition="raw_cut")


<Figure size 1000x600 with 0 Axes>

In [2]:
# =========================================== I. PREPROCESSING ===========================================
import plot
from scipy.ndimage import uniform_filter
# Plot the raw spectrum.
plot.single_gcims_plot(spectrum=spectrum,
                       plot_dir=output_dir,
                       condition="raw",
                       peak_cords=None)

spectrum.rip_ms = 1
# Baseline correction - removing the noisy background by subtracting the mean of an empty area.
spectrum.sub_first_rows(50)
spectrum.cut_dt(start=1.02, stop=1.2)
spectrum.cut_rt(start=100, stop=140)

#uniform_filter_size= (11, 11)
#spectrum.values = uniform_filter(input=spectrum.values, size=uniform_filter_size)
#from scipy.ndimage import gaussian_filter
#spectrum.values = gaussian_filter(spectrum.values, sigma=1.0)

# Smoothing the spectrum with the Savitzky–Golay filter.
spectrum.savgol(window_length=10, polyorder=4)

# Remove negative values after applying the smooth filter.
spectrum.values[spectrum.values < 0] = 0.0

# Plot the processed spectrum.

plot.single_gcims_plot(spectrum=spectrum,
                       plot_dir=output_dir,
                       condition="processed",
                       peak_cords=None)


<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

In [ ]:
import numpy as np
np.percentile(spectrum.values, [99.5])

# Inspect some tree parameters here ...
print(f"SPECTRUM INFOS")
print(f"Type: {type(spectrum.values)}")
print(f"Element Type: {type(spectrum.values[0, 0])}")
print("================================================")
print(f"Shape: {spectrum.values.shape}")
num_entries = spectrum.values.shape[0] * spectrum.values.shape[1]
num_non_0_entries = len(np.flatnonzero(spectrum.values))
print(f"Total Entries: {num_entries}")
print(f"Non-0 Enties: {num_non_0_entries}")
print(f"0 Enties: {num_entries - num_non_0_entries}")
print("================================================")
print(f"Max Value: {spectrum.values.max()}")
print(f"Min Value: {spectrum.values.min()}")
print("================================================")
over_50_values = len(np.where(non_normalized_spectrum.values >= 25)[0])
print(f"Total Values: {num_entries}")
print(f"Values > 50: {over_50_values}")
print(f"Values > 50 in % {(over_50_values/num_entries)*100}")
print(f"Percentile Value: {100 - (over_50_values/num_entries)*100}")



TypeError: 'findpeaks' object is not subscriptable